# 02 - VIX calibration and DCC-GARCH benchmark

Calibrates the Hawkes stress signals and the DCC-GARCH(1,1) benchmark to the VIX (Table 4) and exports the data of Figures 5-6.

Author: Luca Persia (2026), Università della Svizzera italiana, Lugano, Switzerland

In [ ]:
%pip install arch statsmodels

In [ ]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Return the repository root, i.e. the first folder above `start` containing both data/ and src/."""
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "src").is_dir():
            return p
    raise FileNotFoundError(f"Repository root not found above {start}")


ROOT_DIR    = find_repo_root()
RAW_DIR     = ROOT_DIR / "data" / "raw"
EVENTS_DIR  = ROOT_DIR / "data" / "events"
RESULTS_DIR = ROOT_DIR / "data" / "results"
for d in (RAW_DIR, EVENTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from arch import arch_model
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# Observation windows of the paper and input files.
WINDOWS = {
    "GFC": {
        "prices":    RAW_DIR / "daily_prices_GFC_2008.csv",
        "intensity": RESULTS_DIR / "daily_intensity_decomp.csv",
        "start": "2007-06-01", "end": "2011-01-01",
    },
    "COVID": {
        "prices":    RAW_DIR / "daily_prices_COVID_2020.csv",
        "intensity": RESULTS_DIR / "daily_intensity_decomp_covid.csv",
        "start": "2019-06-01", "end": "2024-01-01",
    },
}
FILE_VIX = RAW_DIR / "options_signals.csv"

# Stress signals of Table 4.
SIGNALS = {
    "lambda_uni": "Univariate Hawkes",
    "lambda_tot": "Multivariate Marked Hawkes",
    "dcc_vol":    "DCC-GARCH(1,1)",
}

In [ ]:
# DCC-GARCH(1,1)

def fit_univariate_garch(returns: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Fit a GARCH(1,1) to each column of `returns` and return conditional volatilities and standardized residuals."""
    T, m = returns.shape
    cond_vol = np.zeros((T, m))
    std_resid = np.zeros((T, m))
    for i, col in enumerate(returns.columns):
        # Returns in percent for numerical stability, as usual with `arch`.
        res = arch_model(returns[col].to_numpy() * 100, vol="Garch", p=1, q=1, mean="Constant").fit(disp="off")
        cond_vol[:, i] = res.conditional_volatility / 100
        std_resid[:, i] = res.resid / res.conditional_volatility
    return cond_vol, std_resid


def dcc_path(a: float, b: float, Z: np.ndarray, Q_bar: np.ndarray):
    """Yield the DCC correlation matrices R_t of Engle (2002)."""
    Q = Q_bar.copy()
    for t in range(Z.shape[0]):
        if t > 0:
            z = Z[t - 1][:, None]
            Q = (1 - a - b) * Q_bar + a * (z @ z.T) + b * Q
        q = np.sqrt(np.diag(Q))
        yield Q / np.outer(q, q)


def dcc_negloglik(params: np.ndarray, Z: np.ndarray, Q_bar: np.ndarray) -> float:
    """Negative correlation log-likelihood of the DCC model (up to constants)."""
    a, b = params
    m = Z.shape[1]
    nll = 0.0
    for t, R in enumerate(dcc_path(a, b, Z, Q_bar)):
        R = R + np.eye(m) * 1e-8
        try:
            nll += np.log(np.linalg.det(R)) + Z[t] @ np.linalg.solve(R, Z[t])
        except np.linalg.LinAlgError:
            return 1e10
    return 0.5 * nll


def fit_dcc_garch(returns: pd.DataFrame) -> pd.Series:
    """Two-step DCC-GARCH(1,1); return the annualized volatility (%) of the equally weighted portfolio."""
    cond_vol, Z = fit_univariate_garch(returns)
    Q_bar = np.corrcoef(Z.T)

    # Stationarity: a, b > 0 and a + b < 1.
    cons = ({"type": "ineq", "fun": lambda x: 0.999 - x[0] - x[1]},)
    a, b = minimize(dcc_negloglik, x0=[0.05, 0.5], args=(Z, Q_bar),
                    constraints=cons, bounds=((0.001, 0.998), (0.001, 0.998))).x

    w = np.ones(returns.shape[1]) / returns.shape[1]
    port_vol = np.array([np.sqrt(w @ np.diag(cond_vol[t]) @ R @ np.diag(cond_vol[t]) @ w)
                         for t, R in enumerate(dcc_path(a, b, Z, Q_bar))])
    return pd.Series(port_vol * np.sqrt(252) * 100, index=returns.index, name="dcc_vol")

In [ ]:
# VIX CALIBRATION

def calibrate_to_vix(vix: pd.Series, signal: pd.Series) -> dict:
    """OLS of the VIX on the standardized signal (Eq. 13) and the statistics of Table 4."""
    z = (signal - signal.mean()) / signal.std()
    fit = sm.OLS(vix, sm.add_constant(z)).fit()
    return {
        "R2 (%)":      100 * fit.rsquared,
        "Correlation": np.corrcoef(vix, fit.fittedvalues)[0, 1],
        "RMSE":        np.sqrt(np.mean(fit.resid ** 2)),
        "Slope":       fit.params.iloc[1],
        "t-stat":      fit.tvalues.iloc[1],
    }

In [ ]:
# TABLE 4 AND DATA OF FIGURES 5-6

vix_all = pd.read_csv(FILE_VIX, parse_dates=["date"])[["date", "VIX"]].dropna()
rows = []

for name, cfg in WINDOWS.items():

    # Hawkes signals and the sectors of the multivariate fit.
    hawkes = pd.read_csv(cfg["intensity"], parse_dates=["date"])
    sectors = [c.upper() for c in hawkes.columns if c not in ("date", "lambda_tot", "lambda_uni")]

    # DCC-GARCH on every trading day of the window, for the same sectors.
    prices = pd.read_csv(cfg["prices"], parse_dates=["date"])
    returns = (prices.pivot(index="date", columns="ticker", values="return")
                     .loc[cfg["start"]:cfg["end"], sectors]
                     .dropna())
    dcc_vol = fit_dcc_garch(returns)

    # Common sample: days with VIX, Hawkes signal and DCC value.
    df = (vix_all.merge(hawkes[["date", "lambda_tot", "lambda_uni"]], on="date")
                 .merge(dcc_vol.reset_index(), on="date")
                 .rename(columns={"VIX": "vix"})
                 .sort_values("date"))

    out = df[["date", "vix", "lambda_tot", "lambda_uni", "dcc_vol"]].copy()
    out["date"] = out["date"].dt.strftime("%Y-%m-%d")
    out.to_csv(RESULTS_DIR / f"vola_plot_{name}.csv", index=False)

    for col, label in SIGNALS.items():
        rows.append({"Window": name, "Model": label, **calibrate_to_vix(df["vix"], df[col])})

table4 = pd.DataFrame(rows)
table4.to_csv(RESULTS_DIR / "vix_calibration.csv", index=False)

# Rows ready for the LaTeX table.
for _, r in table4.iterrows():
    print(f"{r.Window:6s}{r.Model:<27s}& {r['R2 (%)']:.2f} & {r.Correlation:.3f} & {r.RMSE:.3f} & {r.Slope:.3f} & {r['t-stat']:.2f} \\\\")